## Week 10 and 11 Assignment - DATASCI200 Introduction to Data Science Programming, UC Berkeley MIDS

Write code in this Jupyter Notebook to solve the following problems. Please upload this **Notebook** with your solutions to gradescope. 

Assignment due date: 11:59PM PT the night before the Week 12 Live Session. Do **NOT** push/upload the data file. 

## Objectives

* Analyze and glean insights from a real dataset using pandas
* Apply pandas for exploratory analysis, information gathering, and discovery
* Demonstrate cleaning data and answering questions

## Dataset

You are to analyze campaign contributions to the 2016 U.S. presidential primary races made in California. Use the csv file located here: https://drive.google.com/file/d/1Lgg-PwXQ6TQLDowd6XyBxZw5g1NGWPjB/view?usp=sharing. You should download and save this file in the same folder as this notebook is stored.  This file originally came from the U.S. Federal Election Commission (https://www.fec.gov/).

**DO NOT PUSH THIS FILE TO YOUR GITHUB REPO!**

- Best practice is to not have DATA files in your code repo. As shown below, the default load is outside of the folder this notebook is in. If you change the folder where the file is stored please update the first cell!
- If you do accidentally push the file to your github repo - follow the directions here to fix it: https://docs.google.com/document/d/15Irgb5V5G7pKPWgAerH7FPMpKeQRunbNflaW-hR2hTA/edit?usp=sharing

Documentation for this data can be found here: https://drive.google.com/file/d/11o_SByceenv0NgNMstM-dxC1jL7I9fHL/view?usp=sharing

## General Guidelines:

- This is a **real** dataset and so it may contain errors and other pecularities to work through
- This dataset is ~218mb, which will take some time to load (and probably won't load in Google Sheets or Excel)
- If you make assumptions, annotate them in your responses
- While there is one code/markdown cell positioned after each question as a placeholder, some of your code/responses may require multiple cells
- Double-click the markdown cells that say for example **1a answer here:** to enter your written answers. If you need more cells for your written answers, make them markdown cells (rather than code cells)
- This homework assignment is not autograded because of the variety of responses one could give. 
  - Please upload this notebook to the autograder page and the TAs will manually grade it. 
  - Ensure that each cell is run and outputs your answer for ease of grading! 
  - Highly suggest to do a `restart & run all` before uploading your code to ensure everything runs and outputs correctly.
  - Answers without code (or code that runs) will be given 0 points.
- **This is meant to simulate real world data so you will have to do some external research to determine what some of the answers are!** 

## Data Questions

You are working for a California state-wide election campaign. Your boss wants you to examine historic 2016 election contribution data to see what zipcodes are more supportive of fundraising for your candidate. 

Your boss asks you to filter out some of the records:
- Only use primary 2016 contribution data (more like how your race is).
- Concentrate on Bernie Sanders as a candidate (most a like your candidate)

The questions your boss wants answered is:
- Which zipcode (5-digit zipcode) had the highest count of contributions and the most dollar amount?
- What day(s) of the month do most people donate?

## Setup

Run the cell below as it will load the data into a pandas dataframe named `contrib`. Note that a custom date parser is defined to speed up loading. If Python were to guess the date format, it would take even longer to load.

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

# These commands below set some options for pandas and to have matplotlib show the charts in the notebook
pd.set_option('display.max_rows', 1000)
pd.options.display.float_format = '{:,.2f}'.format

# Define a date parser to pass to read_csv
d = lambda x: datetime.strptime(x, '%d-%b-%y')

# Load the data
# We have this defaulted to the folder OUTSIDE of your repo - please change it as needed
contrib = pd.read_csv('P00000001-CA.csv', index_col=False, parse_dates=['contb_receipt_dt'], date_format='%d-%b-%y')

# Note - for now, it is okay to ignore the warning about mixed types. 

/var/folders/96/1rsv4bd93l5d2w8p58cmv_0r0000gn/T/ipykernel_15136/385358853.py:14: DtypeWarning: Columns (0: contbr_zip, 1: receipt_desc, 2: memo_cd) have mixed types. Specify dtype option on import or set low_memory=False.
  contrib = pd.read_csv('P00000001-CA.csv', index_col=False, parse_dates=['contb_receipt_dt'], date_format='%d-%b-%y')


***
## 1. Initial Data Checks (50 points)

First we will take a preliminary look at the data to check that it was loaded correctly and contains the info we need.

The questions to answer at the end of this section:
- Do we have the correct # of columns and rows. 
- Do the records contain data for the questions we want to answer 
- What columns are important? 
- What columns can be dropped?
- What are the data problems?

**1a.** Print the *shape* of the data. Does this match the expectation? (2 points)

In [ ]:
# 1a YOUR CODE HERE
print(contrib.shape)
#yes it has the correctn umber of columns and rows

(1125659, 18)


- **1a answer here:** 
it has 18 columns which matches the file, and it has 1125659 rows which is one less then the file but that makes sense due to the header

**1b.** Print a list of column names. Are all the columns included that are in the documentation? (2 points)

In [ ]:
# 1b YOUR CODE HERE
print(contrib.columns)

Index(['cmte_id', 'cand_id', 'cand_nm', 'contbr_nm', 'contbr_city',
       'contbr_st', 'contbr_zip', 'contbr_employer', 'contbr_occupation',
       'contb_receipt_amt', 'contb_receipt_dt', 'receipt_desc', 'memo_cd',
       'memo_text', 'form_tp', 'file_num', 'tran_id', 'election_tp'],
      dtype='str')


- **1b answer here:** 
looks correct, all the columns are there

**1c** Print out the first five rows of the dataset. How do the columns `cand_id`, `cand_nm` and `contbr_st` look? (3 points)

In [ ]:
# 1c YOUR CODE HERE
print(contrib.head(5))
#

     cmte_id    cand_id                  cand_nm          contbr_nm  \
0  C00575795  P00003392  Clinton, Hillary Rodham         AULL, ANNE   
1  C00575795  P00003392  Clinton, Hillary Rodham  CARROLL, MARYJEAN   
2  C00575795  P00003392  Clinton, Hillary Rodham   GANDARA, DESIREE   
3  C00577130  P60007168         Sanders, Bernard          LEE, ALAN   
4  C00577130  P60007168         Sanders, Bernard   LEONELLI, ODETTE   

     contbr_city contbr_st     contbr_zip            contbr_employer  \
0       LARKSPUR        CA 949,391,913.00                        NaN   
1        CAMBRIA        CA 934,284,638.00                        NaN   
2        FONTANA        CA 923,371,507.00                        NaN   
3      CAMARILLO        CA 930,111,214.00  AT&T GOVERNMENT SOLUTIONS   
4  REDONDO BEACH        CA 902,784,310.00   VERICOR ENTERPRISES INC.   

   contbr_occupation  contb_receipt_amt contb_receipt_dt receipt_desc memo_cd  \
0            RETIRED              50.00       2016-04-26   

- **1c answer here:** 
columns look correct, candidate id is a 9 alphanumeric string, candidate name is a string, and contr_st is the state of the contributor which is CA

**1d.** Print out the values for the column `election_tp`. In your own words, based on the documentation, what information does the `election_tp` variable contain? Do the values in the column match the documentation? (3 points)

In [11]:
# 1d YOUR CODE HERE
print(contrib['election_tp'])

0          P2016
1          P2016
2          P2016
3          P2016
4          P2016
           ...  
1125654    P2016
1125655    P2016
1125656    P2016
1125657    P2016
1125658    P2016
Name: election_tp, Length: 1125659, dtype: str


- **1d answer here:** 
documentation says this is the election typ, which should be in the format EYYYY, which from the out put matches E being P which is the Primary.  YYYY being the year. so P2016, since this is for the primary in 2016

**1e.** Print out the datatypes for all of the columns. What are the datatypes for the `contbr_zip`, `contb_receipt_amt`, `contb_receipt_dt`? (5 points)

In [12]:
# 1e YOUR CODE HERE
print(contrib.dtypes)


cmte_id                         str
cand_id                         str
cand_nm                         str
contbr_nm                       str
contbr_city                     str
contbr_st                       str
contbr_zip                   object
contbr_employer                 str
contbr_occupation               str
contb_receipt_amt           float64
contb_receipt_dt     datetime64[us]
receipt_desc                    str
memo_cd                         str
memo_text                       str
form_tp                         str
file_num                      int64
tran_id                         str
election_tp                     str
dtype: object


- **1e answer here:** 
contbr_zip is object
contb_reciept_amt is a float64
contb_receipt_dt is a datetime64[us]

**1f.** What columns have the most non-nulls?  Would you recommend to drop any columns based on the number of nulls? (5 points)

In [23]:
# 1f YOUR CODE HERE
contrib.isna().sum().sort_values(ascending=False)



receipt_desc         1110614
memo_cd               981391
memo_text             624511
contbr_employer       157902
contbr_occupation      10399
election_tp             1425
contbr_zip                95
contbr_city               26
tran_id                    0
file_num                   0
form_tp                    0
cmte_id                    0
contb_receipt_dt           0
cand_id                    0
contbr_st                  0
contbr_nm                  0
cand_nm                    0
contb_receipt_amt          0
dtype: int64

- **1f answer here:** 
most is the receipt_desc

remove receipt desc, its duplicated in memo text if it is not null.

The rest could have some impact so will be left in


**1g.** A column we know that we want to use is the cand_nm column.  From the documentation each candidate is a unique candidate id also. Check data quality of `cand_id` column to see if it matches `cand_nm` column. Specifically check to ensure our targetted candidate 'Bernard Sanders' always has the same cand_id throughout. Any issues with `cand_nm` matching `cand_id`? (5 points)

In [40]:
# 1g YOUR CODE HERE
display(contrib.query('cand_id == "P60007168" & cand_nm != "Sanders, Bernard"'))
print(contrib[contrib['cand_id']=='P60007168'].groupby(['cand_nm','cand_id']).size().reset_index())
print(contrib[contrib['cand_nm']=='Sanders, Bernard'].groupby(['cand_nm','cand_id']).size().reset_index())

,cmte_id,cand_id,cand_nm,contbr_nm,contbr_city,contbr_st,contbr_zip,contbr_employer,contbr_occupation,contb_receipt_amt,contb_receipt_dt,receipt_desc,memo_cd,memo_text,form_tp,file_num,tran_id,election_tp


            cand_nm    cand_id       0
0  Sanders, Bernard  P60007168  407172
            cand_nm    cand_id       0
0  Sanders, Bernard  P60007168  407172


- **1g answer here:** 

three check, each show that the ID is related to Benard Sanders

**1h.** Another area to check is to make sure all of the records are from California. Check the `contbr_st` column - are there any records outside of California based on `contbr_st`? (5 points)

In [42]:
# 1h YOUR CODE HERE
contrib[contrib['contbr_st']!='CA']


,cmte_id,cand_id,cand_nm,contbr_nm,contbr_city,contbr_st,contbr_zip,contbr_employer,contbr_occupation,contb_receipt_amt,contb_receipt_dt,receipt_desc,memo_cd,memo_text,form_tp,file_num,tran_id,election_tp


- **1h answer here:** 
nothing in the data is not equal to CA in the contrib_ST

**1i.** The next column to check for the analysis is the `tran_id` column. This column could be the primary key so look for duplicates. How many duplicate entries are there? Any pattern for why are there duplicate entries? (5 points)

In [47]:
# 1i YOUR CODE HERE
contrib.groupby(['contbr_nm','contbr_city','contbr_st','contbr_zip','contbr_employer','contbr_occupation'])['tran_id'].nunique().reset_index(name='tran_count')


,contbr_nm,contbr_city,contbr_st,contbr_zip,contbr_employer,contbr_occupation,tran_count
0,"ALERIS, ANNAKIM",SANTA CRUZ,CA,950634048,PAMF,ADMIN,4
1,"HEGER, ASTRID HEPPENSTALL",GLENDALE,CA,912081947,USC,PHYSICIAN,1
2,"MAZEY, ANITA JODELSOHN",LOS ANGELES,CA,"900,642,315.00",SELF-EMPLOYED,MARKETING,1
3,"MAZEY, ANITA JODELSOHN",LOS ANGELES,CA,900642315,SELF-EMPLOYED,MARKETING,1
4,"WILLIAMS, DENA ASLANIAN",SAN FRANCISCO,CA,941161452,COLDWELL BANKER,REAL ESTATE SALES,1
...,...,...,...,...,...,...,...
242740,"ZYKOFSKY, PAUL",SACRAMENTO,CA,"958,114,199.00",PAUL ZYKOFSKY,URBAN PLANNER,2
242741,"ZYLBERT, BARBARA",SARATOGA,CA,950701605,SELF-EMPLOYED,PHYSICIAN,1
242742,"ZYPHUR, MICHAEL",LA MESA,CA,"91,942.00",UNIVERSITY OF MELBOURNE,PROFESSOR,7
242743,"ZYPHUR, MICHAEL",LA MESA,CA,91942,UNIVERSITY OF MELBOURNE,PROFESSOR,1


- **1i answer here:** 

assuming that the contributor is unique based on a combination of all contributor categorization data (i.e. multiple people with the same name don't work at the same company, live in the same zip,city). seems like formatting of the zipcode is the most common

**1j.** Another column to check is the `contb_receipt_amt` that shows the donation amounts. How many negative donations are included? What do negative donations mean? Please show at least pull a few rows to look at the records with negative donations. Do these records match with the expectation of why a negative donation would happen? (5 points)

In [55]:
# 1j YOUR CODE HERE
print(contrib[contrib.contb_receipt_amt<0].count())

print(contrib[contrib.contb_receipt_amt<0].groupby(['memo_text']).size().reset_index(name = 'count').sort_values(ascending=False,by='count'))


cmte_id              11896
cand_id              11896
cand_nm              11896
contbr_nm            11896
contbr_city          11896
contbr_st            11896
contbr_zip           11893
contbr_employer       3213
contbr_occupation     3317
contb_receipt_amt    11896
contb_receipt_dt     11896
receipt_desc         11054
memo_cd               3031
memo_text             3157
form_tp              11896
file_num             11896
tran_id              11896
election_tp          11746
dtype: int64
                                            memo_text  count
33                           REDESIGNATION TO GENERAL   1324
32                   REDESIGNATION TO CRUZ FOR SENATE    544
26                            REATTRIBUTION TO SPOUSE    533
1                              * HILLARY VICTORY FUND    399
21                                       CHARGED BACK    228
34              REDESIGNATION TO PRESIDENTIAL GENERAL     41
24                                 NSF/RETURNED CHECK     11
37      REFUN

- **1j answer here:**
11896 negative contributions
The contribution is moved from funding the primary to the general election is the most common
Negative contribution is due to reattribution of funds, or pull back of funds, which makes sense. 

**1k.** One more column to look at is the date of donation column. Are there any dates outside of the primary period (defined as 1 Jan 2014 to 7 June 2016)? Are the dates well-formatted for our analysis? (5 points)

In [58]:
# 1k YOUR CODE HERE
contrib['contb_receipt_dt'].describe()
contrib.contb_receipt_dt[~contrib['contb_receipt_dt'].between('2014-01-01','2016-06-07')]


9932      2013-11-05
9994      2013-11-05
14673     2016-06-30
14682     2016-07-20
14697     2016-07-14
             ...    
1123992   2016-08-02
1123993   2016-08-09
1123994   2016-08-31
1123995   2016-08-31
1123996   2016-07-06
Name: contb_receipt_dt, Length: 451372, dtype: datetime64[us]

- **1k answer here:**
there are 451372 dates outside of the receipt time period.  The dates are not well formatted, we have to assume 16 is in 2016 and not 1916 or 1816.


**1l.** Finally, answer the initial questions in the cells below (5 points)

**1l.1** Do we have the correct # of columns and rows.

- **1l.1 answer here:**
yes

**1l.2** Do the records contain data for the questions we want to answer?

- **1l.2 answer here:**
yes

**1l.3** What columns are important?

- **1l.3 answer here:** 
everything other then reciept_desc

**1l.4** What columns can be dropped?

- **1l.4 answer here:** 
reciept_desc

**1l.5** What are the data problems?

- **1l.5 answer here:**
dates are out of the primary value, will need to be evaluated.
reciept description is incomplete,
formatting of zipcodes is off
assumes that all the contributor columns create a unique idea, which is cannot be proven with the data we have.
dates don't match the data dictionary which says DD-MMM-YYY, instead its dd-mmm-yy


**1l.6** List any assumptions so far:

- **1l.6 answer here:**
date are in the format dd-Mon-yy, the assumption is that yy is all 20yy.
That this is accurate information

***
## 2. Data filtering and data quality fixes (30 points)

Now that we have a basic understanding of the data, let's filter out the records we don't need and fix the data. Do each of the following problems sequentially so that at the end the dataframe is filtered and cleaned for the analysis. (that is, use the dataframe answer for 2a to start 2b etc)

**2a.** From the dataset filter out (remove) any election_tp not in the primary election for 2016. Also filter for the primary dates (defined as 1 Jan 2014 to 7 June 2016). Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [62]:
# 2a YOUR CODE HERE

a = contrib[(contrib['election_tp']=='P2016') & (contrib['contb_receipt_dt'].between('2014-01-01','2016-06-07'))]
print(a.shape)

(668864, 18)


**2b.** From the dataset filter out (remove) any candidate that is not Bernie Sanders. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [63]:
# 2b YOUR CODE HERE
b = a[a.cand_id=='P60007168']
print(b.shape)

(379284, 18)


**2c.** The `contbr_zip` column is not formatted well for our analysis. Make a new zipcode column that is the five-digit zipcodes. Filter out any records outside of California based on the zipcode. Print/show the shape of the dataframe after the filtering is complete. (10 points).

- You will have to research what the valid 5-digit zipcodes for California are!

In [65]:
# 2c YOUR CODE HERE
b['correct_zip'] = b.contbr_zip.apply(lambda x: str(x)[:5])
c = b[b.correct_zip.between('90001','96162')]
print(c.shape)

(379265, 19)


**2d.** The receipt amount column has negative donations. After talking with your team, a decision was made that the best course of action is to remove these negative values so that the donation count and amount is more accurate. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [66]:
# 2d YOUR CODE HERE
d = c[c.contb_receipt_amt>=0]
print(d.shape)


(376973, 19)


**2e.** From the dataset drop any columns that won't be used in the analysis. Print/show the shape of the dataframe after the dropping is complete. What columns did you drop and why? (5 points)

In [69]:
# 2e YOUR CODE HERE
e = d.drop(columns=['receipt_desc'])

- **2e answer here:**
just receipt description, its information is missing for most columns, and it is duplicated in the memo_desc

In [70]:
e

,cmte_id,cand_id,cand_nm,contbr_nm,contbr_city,contbr_st,contbr_zip,contbr_employer,contbr_occupation,contb_receipt_amt,contb_receipt_dt,memo_cd,memo_text,form_tp,file_num,tran_id,election_tp,correct_zip
3,C00577130,P60007168,"Sanders, Bernard","LEE, ALAN",CAMARILLO,CA,"930,111,214.00",AT&T GOVERNMENT SOLUTIONS,SOFTWARE ENGINEER,40.00,2016-03-04,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1077404,VPF7BKWA097,P2016,93011
4,C00577130,P60007168,"Sanders, Bernard","LEONELLI, ODETTE",REDONDO BEACH,CA,"902,784,310.00",VERICOR ENTERPRISES INC.,PHARMACIST,35.00,2016-03-05,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1077404,VPF7BKX3MB3,P2016,90278
5,C00577130,P60007168,"Sanders, Bernard","LEONELLI, ODETTE",REDONDO BEACH,CA,"902,784,310.00",VERICOR ENTERPRISES INC.,PHARMACIST,100.00,2016-03-06,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1077404,VPF7BKYBXV4,P2016,90278
6,C00577130,P60007168,"Sanders, Bernard","LEOPARD, PATTI",VISTA,CA,"920,842,849.00",ONSITE ENERGY CORPORATION,PROJECT MANAGER,25.00,2016-03-04,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1077404,VPF7BKW04C1,P2016,92084
8,C00577130,P60007168,"Sanders, Bernard","LEPKE, KELLY",WESTMINSTER,CA,"926,833,846.00",NONE,NOT EMPLOYED,10.00,2016-03-05,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1077404,VPF7BKX3H59,P2016,92683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1121696,C00577130,P60007168,"Sanders, Bernard","SIMONE, ARLENE",ENCINO,CA,914361655,SELF EMPLOYED-EMPLOYED,ENTERTASINER,50.00,2016-04-23,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1079445,VPF7BP5TFS6,P2016,91436
1121698,C00577130,P60007168,"Sanders, Bernard","SIMONE, ARLENE",ENCINO,CA,914361655,SELF EMPLOYED-EMPLOYED,ENTERTASINER,100.00,2016-04-27,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1079445,VPF7BP86GH6,P2016,91436
1121703,C00577130,P60007168,"Sanders, Bernard","ROBINSON, KATHARINE",FULLERTON,CA,928313414,CITY OF LOS ANGELES,MANAGEMENT ASSISTANT,15.00,2016-04-15,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1079445,VPF7BNXXDY5,P2016,92831
1121704,C00577130,P60007168,"Sanders, Bernard","ROBINSON, KATHARINE",FULLERTON,CA,928313414,CITY OF LOS ANGELES,MANAGEMENT ASSISTANT,15.00,2016-04-23,NaN,* EARMARKED CONTRIBUTION: SEE BELOW,SA17A,1079445,VPF7BP64DH7,P2016,92831


**2f.** List any assumptions that you made up to this point or if there is any filtering you think is needed and why:

NOTE: You can look to see if there are any duplicate rows also - please write an assumption on what you would do with these!

- **2f answer here:**

assuming a combination of all the contributor descriptive columns would create a unique contributor, this does not have to be the case
should filter to only include SA17A


***
## 3. Answering the questions (20 points)

Now that the data is cleaned and filterd - let's answer the two questions from your boss! That is use the dataframe from 2e above that has done all of the cleaning & filtering steps!

**3a.** Which zipcode had the highest count of contributions and the most dollar amount? (10 points)

In [80]:
# 3a YOUR CODE HERE
print(e.groupby('correct_zip').size().reset_index(name='count').sort_values(ascending=False,by='count').head(1))
print(e.groupby('correct_zip')['contb_receipt_amt'].sum().reset_index(name='count').sort_values(ascending=False,by='count').head(1))


     correct_zip  count
1055       94110   3799
     correct_zip      count
1055       94110 284,398.05


- **3a answer here:** 
94110 had the most contributions
94110 also had the highest amount 284,398.05


**3b.** What day(s) of the month do most people donate? (10 points)

In [84]:
# 3b YOUR CODE HERE
print(e.groupby([e.contb_receipt_dt.dt.day]).size().reset_index(name='count').sort_values(ascending=False,by='count').head(1))
print(e.groupby([e.contb_receipt_dt.dt.day_name()]).size().reset_index(name='count').sort_values(ascending=False,by='count').head(1))

    contb_receipt_dt  count
28                29  21838
  contb_receipt_dt  count
6        Wednesday  75727


- **3b answer here:** 
I don't know what you mean here, is this the numerical or the name of the day???
they donate most often on the 29th, and on Wednesday.

## If you have feedback for this homework, please submit it using the link below:

http://goo.gl/forms/74yCiQTf6k